<a href="https://colab.research.google.com/github/nimraa9090/AI-projects/blob/main/ANN_DL_PRJ_Solution.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Medical Image Classification — AneRBC Dataset
**Name:** \<Your Name\>
**SRN:** \<Your SRN\>

Custom CNNs (3/4/5 layers), Transfer Learning, and Explainable AI (Grad-CAM) for blood smear anemia classification.


In [2]:
# Name: <NIMRA SAEED>
# SRN: <303231009>
# Project: Medical Image Classification - AneRBC Dataset


## Task 0: Setup — Drive & Git

In [3]:
from google.colab import drive
drive.mount('/content/drive')


MessageError: Error: credential propagation was unsuccessful

In [ ]:
from google.colab import userdata

GITHUB_USERNAME = "nimraa9090"
GITHUB_PAT = userdata.get('GITHUB_PAT')   # Add this secret via the key icon in Colab's left sidebar.
                                            # NEVER hardcode your token directly in a cell.
REPO_URL = "https://github.com/nimraa9090/ANN-DL-PRJ"

!git init
!git config --global user.email "nishaarmin@gmail.com"
!git config --global user.name "nimra9090"

pat_repo_url = f"https://{GITHUB_USERNAME}:{GITHUB_PAT}@{REPO_URL.replace('https://', '')}"
!git remote remove origin 2>/dev/null
!git remote add origin {pat_repo_url}
!git remote -v


In [ ]:
import os, cv2
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, classification_report,
                              confusion_matrix, ConfusionMatrixDisplay)

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using:", device)


## Task 1: Loading & Cleaning the Dataset

### 1.1 Dataset location

Dataset manually downloaded and placed in Google Drive (see README for source link / download instructions).

In [ ]:
BASE_PATH = "/content/drive/MyDrive/anerbc_dataset/  AneRBC dataset/AneRBC_dataset"

paths = [
    ("AneRBC-I/Anemic_individuals/Original_images", 0),
    ("AneRBC-I/Healthy_individuals/Original_images", 1),
    ("AneRBC-II/Anemic_individuals/Original_images", 0),
    ("AneRBC-II/Healthy_individuals/Original_images", 1),
]


### 1.2 Data validation: corrupted files, label mapping, class distribution

In [ ]:
def validate_and_collect(base_path, paths):
    """
    Validate images and collect file paths with labels.

    Args:
        base_path (str): root dataset directory.
        paths (list[tuple]): list of (subfolder, label) pairs.

    Returns:
        data (list[str]): valid image file paths.
        labels (list[int]): corresponding labels.
        corrupted (list[str]): paths that failed to load.

    Assumes: subfolders exist under base_path; images are readable by OpenCV.
    """
    data, labels, corrupted = [], [], []

    for sub_path, label in paths:
        full_path = os.path.join(base_path, sub_path)
        if not os.path.exists(full_path):
            print("Missing folder:", full_path)
            continue

        for fname in os.listdir(full_path):
            fpath = os.path.join(full_path, fname)
            if not os.path.isfile(fpath):
                continue
            img = cv2.imread(fpath)
            if img is None or img.size == 0:
                corrupted.append(fpath)
                continue
            data.append(fpath)
            labels.append(label)

    return data, labels, corrupted

data, labels, corrupted = validate_and_collect(BASE_PATH, paths)

print("Total valid images:", len(data))
print("Corrupted/unreadable images:", len(corrupted))
for c in corrupted[:10]:
    print(" -", c)

class_map = {0: "Anemic", 1: "Healthy"}
unique, counts = np.unique(labels, return_counts=True)
for u, c in zip(unique, counts):
    print(f"Class {u} ({class_map[u]}): {c} images")

plt.bar([class_map[u] for u in unique], counts, color=["salmon", "skyblue"])
plt.title("Class Distribution")
plt.ylabel("Count")
plt.show()


### 1.3 Preprocessing pipeline (resize + normalization)

In [ ]:
IMG_SIZE = 128

# Normalize with ImageNet stats so pretrained models (Task 3) also work consistently
transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                          std=[0.229, 0.224, 0.225])
])


In [ ]:
class RBCDataset(Dataset):
    """
    PyTorch Dataset for AneRBC blood smear images.

    Args:
        image_paths (list[str]): paths to validated image files.
        labels (list[int]): integer class labels.
        transform (callable): torchvision transform pipeline.

    Assumes all paths have already been validated (no corrupted files).
    """
    def __init__(self, image_paths, labels, transform=None):
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img = cv2.imread(self.image_paths[idx])
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        if self.transform:
            img = self.transform(img)
        label = torch.tensor(self.labels[idx], dtype=torch.long)
        return img, label


### 1.4 Stratified train/val/test split + dataloaders (deterministic seed)

In [ ]:
train_paths, temp_paths, train_labels, temp_labels = train_test_split(
    data, labels, test_size=0.3, stratify=labels, random_state=SEED
)
val_paths, test_paths, val_labels, test_labels = train_test_split(
    temp_paths, temp_labels, test_size=0.5, stratify=temp_labels, random_state=SEED
)

print(f"Train: {len(train_paths)}, Val: {len(val_paths)}, Test: {len(test_paths)}")

train_dataset = RBCDataset(train_paths, train_labels, transform=transform)
val_dataset   = RBCDataset(val_paths, val_labels, transform=transform)
test_dataset  = RBCDataset(test_paths, test_labels, transform=transform)

BATCH_SIZE = 32
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)


### 1.5 Commit & push

In [ ]:
%%writefile .gitignore
drive/
data/raw/
*.jpg
*.jpeg
*.png
*.bmp
*.tif
__pycache__/
.ipynb_checkpoints/

In [ ]:
# Remove the accidentally-tracked Drive/image files from git's index (keeps the actual files, just untracks them)
!git rm -r --cached drive 2>/dev/null
!git status

In [ ]:
!git remote -v

In [ ]:
!git add .
!git commit -m "Task1: data validation, preprocessing, stratified train/val/test split"
!git push


In [ ]:
!git push -u origin master

## Task 2: Custom Deep CNN (3, 4, and 5 layers)

### 2.1 Model architectures

In [ ]:
class CNN3Layer(nn.Module):
    """3 conv-block CNN: Conv-ReLU-Pool x3 + FC head with dropout."""
    def __init__(self, num_classes=2):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3,16,3,padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16,32,3,padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32,64,3,padding=1), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.3),
            nn.Linear(64*16*16, 128), nn.ReLU(),
            nn.Linear(128, num_classes)
        )
    def forward(self, x):
        return self.classifier(self.features(x))


class CNN4Layer(nn.Module):
    """4 conv-block CNN."""
    def __init__(self, num_classes=2):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3,16,3,padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16,32,3,padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32,64,3,padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64,128,3,padding=1), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.4),
            nn.Linear(128*8*8, 128), nn.ReLU(),
            nn.Linear(128, num_classes)
        )
    def forward(self, x):
        return self.classifier(self.features(x))


class CNN5Layer(nn.Module):
    """5 conv-block CNN."""
    def __init__(self, num_classes=2):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3,16,3,padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16,32,3,padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32,64,3,padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64,128,3,padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(128,256,3,padding=1), nn.ReLU(), nn.AdaptiveAvgPool2d(4),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.5),
            nn.Linear(256*4*4, 128), nn.ReLU(),
            nn.Linear(128, num_classes)
        )
    def forward(self, x):
        return self.classifier(self.features(x))


### 2.2 Train & validate (with learning curves)

In [ ]:
def train_model(model, train_loader, val_loader, epochs=10, lr=1e-3):
    """
    Train a model, tracking train/val loss per epoch.

    Args:
        model (nn.Module): model to train.
        train_loader, val_loader (DataLoader): data loaders.
        epochs (int): number of training epochs.
        lr (float): learning rate for Adam optimizer.

    Returns:
        model (nn.Module): trained model.
        history (dict): train_loss and val_loss per epoch.
    """
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    history = {"train_loss": [], "val_loss": []}

    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            out = model(x)
            loss = criterion(out, y)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        train_loss = total_loss / len(train_loader)

        model.eval()
        val_loss = 0
        with torch.no_grad():
            for x, y in val_loader:
                x, y = x.to(device), y.to(device)
                out = model(x)
                val_loss += criterion(out, y).item()
        val_loss /= len(val_loader)

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        print(f"Epoch {epoch+1}/{epochs} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

    return model, history


def plot_learning_curve(history, title):
    """Plot train vs val loss curves."""
    plt.plot(history["train_loss"], label="Train Loss")
    plt.plot(history["val_loss"], label="Val Loss")
    plt.title(title); plt.xlabel("Epoch"); plt.ylabel("Loss")
    plt.legend(); plt.show()


### 2.3 Test, evaluate (precision/recall/F1, classification report, confusion matrix)

In [ ]:
def evaluate_model(model, loader, class_names=("Anemic", "Healthy")):
    """
    Evaluate a trained model on a data loader.

    Args:
        model (nn.Module): trained model.
        loader (DataLoader): evaluation data loader.
        class_names (tuple): display names for classes.

    Returns:
        acc (float): accuracy score. Also prints classification report
        and plots a confusion matrix.
    """
    model.eval()
    preds, true = [], []
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            out = model(x)
            _, p = torch.max(out, 1)
            preds.extend(p.cpu().numpy())
            true.extend(y.numpy())

    acc = accuracy_score(true, preds)
    print("Accuracy:", acc)
    print(classification_report(true, preds, target_names=class_names))

    cm = confusion_matrix(true, preds)
    ConfusionMatrixDisplay(cm, display_labels=class_names).plot(cmap="Blues")
    plt.show()
    return acc


results = {}
for name, ModelClass in [("CNN3", CNN3Layer), ("CNN4", CNN4Layer), ("CNN5", CNN5Layer)]:
    print(f"\n===== Training {name} =====")
    model = ModelClass()
    model, history = train_model(model, train_loader, val_loader, epochs=10)
    plot_learning_curve(history, f"{name} Learning Curve")
    acc = evaluate_model(model, test_loader)
    results[name] = {"model": model, "acc": acc}

best_custom_name = max(results, key=lambda k: results[k]["acc"])
best_custom_model = results[best_custom_name]["model"]
print("Best custom model:", best_custom_name)


### 2.4 Commit & push

In [ ]:
!git add .
!git commit -m "Task2: 3/4/5-layer CNNs trained and evaluated"
!git push


## Task 3: Pretrained CNNs (MobileNet, ResNet, DenseNet) — Fine-tune Top Layers

Backbone is frozen; only the new classifier head is trainable (documented per model below).

In [ ]:
import torchvision.models as models

def build_transfer_model(name, num_classes=2):
    """
    Loads a pretrained backbone, freezes it, replaces the classifier head.

    Args:
        name (str): one of "mobilenet", "resnet", "densenet".
        num_classes (int): number of output classes.

    Returns:
        nn.Module with frozen backbone and a new trainable head.
    """
    if name == "mobilenet":
        m = models.mobilenet_v2(weights="IMAGENET1K_V1")
        for p in m.parameters(): p.requires_grad = False
        m.classifier[1] = nn.Linear(m.last_channel, num_classes)  # trainable
    elif name == "resnet":
        m = models.resnet18(weights="IMAGENET1K_V1")
        for p in m.parameters(): p.requires_grad = False
        m.fc = nn.Linear(m.fc.in_features, num_classes)  # trainable
    elif name == "densenet":
        m = models.densenet121(weights="IMAGENET1K_V1")
        for p in m.parameters(): p.requires_grad = False
        m.classifier = nn.Linear(m.classifier.in_features, num_classes)  # trainable
    else:
        raise ValueError(name)
    return m


transfer_results = {}
for name in ["mobilenet", "resnet", "densenet"]:
    print(f"\n===== {name} =====")
    tmodel = build_transfer_model(name)
    tmodel, hist = train_model(tmodel, train_loader, val_loader, epochs=5, lr=1e-3)
    plot_learning_curve(hist, f"{name} Learning Curve")
    acc = evaluate_model(tmodel, test_loader)
    transfer_results[name] = {"model": tmodel, "acc": acc}

best_pretrained_name = max(transfer_results, key=lambda k: transfer_results[k]["acc"])
best_pretrained_model = transfer_results[best_pretrained_name]["model"]
print("Best pretrained model:", best_pretrained_name)


### Commit & push

In [ ]:
!git add .
!git commit -m "Task3: transfer learning with MobileNet, ResNet, DenseNet (frozen backbone, trained head)"
!git push


## Task 4: XAI — Grad-CAM on Best Custom + Best Pretrained Model

In [ ]:
!pip install grad-cam -q
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image

def run_gradcam(model, target_layer, image_tensor, orig_img_rgb):
    """
    Run Grad-CAM on a single image and overlay the heatmap.

    Args:
        model (nn.Module): trained model.
        target_layer (nn.Module): convolutional layer to visualize.
        image_tensor (Tensor): preprocessed input tensor (C,H,W).
        orig_img_rgb (np.ndarray): original RGB image resized to IMG_SIZE (H,W,3), uint8.

    Returns:
        np.ndarray: RGB image with Grad-CAM heatmap overlay.
    """
    cam = GradCAM(model=model, target_layers=[target_layer])
    grayscale_cam = cam(input_tensor=image_tensor.unsqueeze(0).to(device))[0]
    vis = show_cam_on_image(orig_img_rgb / 255.0, grayscale_cam, use_rgb=True)
    return vis

# Pick one test sample
sample_path = test_paths[0]
raw = cv2.cvtColor(cv2.imread(sample_path), cv2.COLOR_BGR2RGB)
raw_resized = cv2.resize(raw, (IMG_SIZE, IMG_SIZE))
inp = transform(raw_resized)

# Custom CNN: last conv layer in the feature extractor
cam_custom = run_gradcam(best_custom_model, best_custom_model.features[-3], inp, raw_resized)
plt.imshow(cam_custom); plt.title(f"Grad-CAM: {best_custom_name}"); plt.axis("off"); plt.show()

# Pretrained: last conv layer (adjust attribute per architecture)
if best_pretrained_name == "resnet":
    target_layer = best_pretrained_model.layer4[-1]
else:
    target_layer = best_pretrained_model.features[-1]

cam_pretrained = run_gradcam(best_pretrained_model, target_layer, inp, raw_resized)
plt.imshow(cam_pretrained); plt.title(f"Grad-CAM: {best_pretrained_name}"); plt.axis("off"); plt.show()


### Commit & push

In [ ]:
!git add .
!git commit -m "Task4: Grad-CAM XAI on best custom and pretrained models"
!git push


## Task 5 (separate deliverable)
Write the ≤5000-word critical evaluation report and `README.md` separately and submit/commit them
alongside this notebook (see assignment brief, Task 5).